# Tutorial: CryptoHFTData Workflow

This notebook downloads CryptoHFTData orderbook and trades data with the official SDK through `hftbacktest.data.utils.cryptohftdata`, converts both days into HftBacktest's normalized `.npz` format, derives an end-of-day snapshot from day 1, and runs a small backtest on day 2.

Install the optional dependency first:

```bash
pip install "hftbacktest[cryptohftdata]"
```


In [ ]:
import os
from pathlib import Path

import numpy as np
from numba import njit

from hftbacktest import BacktestAsset, GTX, HashMapMarketDepthBacktest, LIMIT
from hftbacktest.data.utils import cryptohftdata
from hftbacktest.data.utils.snapshot import create_last_snapshot

API_KEY = os.environ.get("CRYPTOHFTDATA_API_KEY")
if API_KEY is None:
    raise RuntimeError("Set CRYPTOHFTDATA_API_KEY before running this notebook.")

EXCHANGE = "binance_futures"
SYMBOL = "BTCUSDT"
DAY1 = "2026-03-01"
DAY2 = "2026-03-02"

WORKDIR = Path("tmp/cryptohftdata_workflow")
WORKDIR.mkdir(parents=True, exist_ok=True)

DAY1_FILE = WORKDIR / "btcusdt_20260301.npz"
DAY2_FILE = WORKDIR / "btcusdt_20260302.npz"
DAY1_EOD_SNAPSHOT = WORKDIR / "btcusdt_20260301_eod.npz"


## Download and Convert Day 1

CryptoHFTData stores historical data in hourly parquet chunks. The SDK handles the hourly downloads internally and returns pandas DataFrames.


In [ ]:
orderbook_day1, trades_day1 = cryptohftdata.download(
    symbol=SYMBOL,
    exchange=EXCHANGE,
    start_date=DAY1,
    end_date=DAY1,
    api_key=API_KEY,
)

_ = cryptohftdata.convert(
    orderbook=orderbook_day1,
    trades=trades_day1,
    output_filename=str(DAY1_FILE),
)

del orderbook_day1, trades_day1
DAY1_FILE


## Download and Convert Day 2

We convert day 2 separately so it can be backtested with the end-of-day snapshot derived from day 1.


In [ ]:
orderbook_day2, trades_day2 = cryptohftdata.download(
    symbol=SYMBOL,
    exchange=EXCHANGE,
    start_date=DAY2,
    end_date=DAY2,
    api_key=API_KEY,
)

_ = cryptohftdata.convert(
    orderbook=orderbook_day2,
    trades=trades_day2,
    output_filename=str(DAY2_FILE),
)

del orderbook_day2, trades_day2
DAY2_FILE


## Build Day 1 End-of-Day Snapshot

The sampled CryptoHFTData orderbook feed is update-based rather than a self-contained full-book snapshot, so we bootstrap day 2 from the book reconstructed over day 1.


In [ ]:
_ = create_last_snapshot(
    [str(DAY1_FILE)],
    tick_size=0.1,
    lot_size=0.001,
    output_snapshot_filename=str(DAY1_EOD_SNAPSHOT),
)

DAY1_EOD_SNAPSHOT


## Run a Small Backtest on Day 2

This is intentionally minimal. It keeps one bid and one ask resting at the current top of book and waits for order responses before proceeding.


In [ ]:
@njit
def demo_market_maker(hbt):
    asset_no = 0
    tick_size = hbt.depth(asset_no).tick_size
    lot_size = hbt.depth(asset_no).lot_size
    order_qty = max(lot_size, np.round(0.002 / lot_size) * lot_size)
    next_order_id = 0

    while hbt.elapse(10_000_000) == 0:
        hbt.clear_inactive_orders(asset_no)

        depth = hbt.depth(asset_no)
        best_bid = depth.best_bid_tick * tick_size
        best_ask = depth.best_ask_tick * tick_size

        orders = hbt.orders(asset_no)
        values = orders.values()
        while values.has_next():
            order = values.get()
            if order.cancellable:
                hbt.cancel(asset_no, order.order_id, False)

        if not hbt.elapse(1_000_000) != 0:
            return False

        bid_order_id = next_order_id
        hbt.submit_buy_order(asset_no, bid_order_id, best_bid, order_qty, GTX, LIMIT, False)
        next_order_id += 1

        ask_order_id = next_order_id
        hbt.submit_sell_order(asset_no, ask_order_id, best_ask, order_qty, GTX, LIMIT, False)
        next_order_id += 1

        if not hbt.wait_order_response(asset_no, ask_order_id, 1_000_000_000):
            return False

    return True


In [ ]:
asset = (
    BacktestAsset()
        .data([str(DAY2_FILE)])
        .initial_snapshot(str(DAY1_EOD_SNAPSHOT))
        .linear_asset(1.0)
        .constant_order_latency(10_000_000, 10_000_000)
        .risk_adverse_queue_model()
        .no_partial_fill_exchange()
        .trading_value_fee_model(-0.00005, 0.0007)
        .tick_size(0.1)
        .lot_size(0.001)
)

hbt = HashMapMarketDepthBacktest([asset])
completed = demo_market_maker(hbt)

print("Completed:", completed)
print("Final timestamp:", hbt.current_timestamp)
print("Final position:", hbt.position(0))
print("Best bid / ask:", hbt.depth(0).best_bid, hbt.depth(0).best_ask)

_ = hbt.close()
